[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filippolonghi/medleydb-mert-instrument-classification/blob/main/notebooks/09_error_analysis_audio_examples.ipynb)


# 09 - Error Analysis and Audio Examples

## Qualitative purpose

This notebook selects representative correct examples and failure cases for the final presentation. It helps turn aggregate metrics into audible evidence: high-confidence mistakes, false positives, false negatives, and classes with weak recall or F1.

## Technical purpose

The notebook reads existing prediction artifacts from isolated single-label or polyphonic multi-label experiments. It does not train models, extract embeddings, modify caches, update checkpoints, change experiment IDs, or write registry rows. Tables and optional audio clips are saved under `results/error_examples/<experiment_id>/`.

## How to use these examples

Use a small number of exported clips in slides or a demo to explain what the model hears when it succeeds or fails. Pair clips with the saved CSV rows so the report can cite the true label, predicted label, confidence, and source metadata.


## What you can change

Set `EXPERIMENT_ID`, `TASK_TYPE`, `N_EXAMPLES_PER_TYPE`, and `EXPORT_AUDIO`. `TASK_TYPE="auto"` detects the prediction format written by the existing single-label and multi-label runners.


In [ ]:
from pathlib import Path
import os
import re
import shutil
import sys

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/medleydb-mert-instrument-classification"))
RUN_ROOT = Path(os.environ.get("RUN_ROOT", "/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"))
MEDLEYDB_ROOT = Path(os.environ.get("MEDLEYDB_ROOT", "/content/drive/MyDrive/medleydb_mert_project/MedleyDB"))
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)
if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = RUN_ROOT / "results"

print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("MedleyDB root:", MEDLEYDB_ROOT)


In [ ]:
from src.data.audio_examples import export_audio_segment, load_isolated_stem_segment, load_mixture_segment

EXPERIMENT_ID = "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02"
TASK_TYPE = "auto"  # auto | single_label | multi_label
SEED = 42
RUN_ID = "latest"
N_EXAMPLES_PER_TYPE = 8
EXPORT_AUDIO = False
TARGET_SAMPLE_RATE = 22050

ERROR_DIR = RESULTS_DIR / "error_examples" / EXPERIMENT_ID
AUDIO_DIR = ERROR_DIR / "audio"
ERROR_DIR.mkdir(parents=True, exist_ok=True)
if EXPORT_AUDIO:
    AUDIO_DIR.mkdir(parents=True, exist_ok=True)
print("Error-analysis output:", ERROR_DIR)


In [ ]:
def find_experiment_dir(experiment_id):
    candidates = [
        RESULTS_DIR / experiment_id,
        RESULTS_DIR / experiment_id / f"seed_{SEED}" / RUN_ID,
    ]
    candidates.extend(sorted((RESULTS_DIR / experiment_id).glob("seed_*/*")) if (RESULTS_DIR / experiment_id).exists() else [])
    for candidate in candidates:
        if (candidate / "predictions.csv").exists():
            return candidate
    raise FileNotFoundError(f"No predictions.csv found for {experiment_id} under {RESULTS_DIR}")


def load_optional_csv(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

EXPERIMENT_DIR = find_experiment_dir(EXPERIMENT_ID)
PREDICTIONS_CSV = EXPERIMENT_DIR / "predictions.csv"
METRICS_JSON = EXPERIMENT_DIR / "test_metrics.json"
REPORT_CSV = EXPERIMENT_DIR / "classification_report.csv"
PER_CLASS_CSV = EXPERIMENT_DIR / "per_class_metrics.csv"
CONFIG_RESOLVED = EXPERIMENT_DIR / "config_resolved.yaml"

predictions = pd.read_csv(PREDICTIONS_CSV)
classification_report = load_optional_csv(REPORT_CSV)
per_class_metrics = load_optional_csv(PER_CLASS_CSV)
print("Predictions:", PREDICTIONS_CSV)
print("Rows:", len(predictions))
display(predictions.head())


In [ ]:
def detect_task_type(frame):
    if TASK_TYPE != "auto":
        return TASK_TYPE
    if {"true_label", "predicted_label"}.issubset(frame.columns):
        return "single_label"
    true_cols = [column for column in frame.columns if column.startswith("true_")]
    pred_cols = [column for column in frame.columns if column.startswith("pred_")]
    if true_cols and pred_cols and {"true_labels", "predicted_labels"}.issubset(frame.columns):
        return "multi_label"
    raise ValueError("Could not infer task type from predictions columns")


def safe_label(value):
    text = "none" if pd.isna(value) or str(value) == "" else str(value)
    return re.sub(r"[^A-Za-z0-9._-]+", "_", text).strip("_")[:80] or "none"


def row_identifier(row):
    for key in ["segment_id", "mixture_id", "track_id"]:
        value = row.get(key, "")
        if not pd.isna(value) and str(value):
            return safe_label(value)
    return f"row_{int(row.name)}"

TASK_TYPE_RESOLVED = detect_task_type(predictions)
print("Task type:", TASK_TYPE_RESOLVED)


In [ ]:
def single_label_tables(frame):
    prob_cols = [column for column in frame.columns if column.startswith("prob_")]
    data = frame.copy()
    data["confidence"] = data.get("probability_predicted_class", data[prob_cols].max(axis=1) if prob_cols else 0.0)
    data["correct_bool"] = data.get("correct", data["true_label"] == data["predicted_label"]).astype(bool)
    errors = data[~data["correct_bool"]].sort_values("confidence", ascending=False)
    correct = data[data["correct_bool"]].sort_values("confidence", ascending=False)
    labels = sorted(set(data["true_label"].dropna().astype(str)) | set(data["predicted_label"].dropna().astype(str)))
    fp_rows = []
    fn_rows = []
    selected = []
    for label in labels:
        fp = data[(data["predicted_label"] == label) & (data["true_label"] != label)].sort_values("confidence", ascending=False)
        fn = data[(data["true_label"] == label) & (data["predicted_label"] != label)].sort_values("confidence", ascending=False)
        fp_rows.append({"class": label, "false_positives": len(fp), "mean_confidence": float(fp["confidence"].mean()) if len(fp) else 0.0})
        fn_rows.append({"class": label, "false_negatives": len(fn), "mean_confidence": float(fn["confidence"].mean()) if len(fn) else 0.0})
        if not fp.empty:
            row = fp.head(1).copy(); row["example_type"] = "false_positive"; row["example_class"] = label; selected.append(row)
        if not fn.empty:
            row = fn.head(1).copy(); row["example_type"] = "false_negative"; row["example_class"] = label; selected.append(row)
    if not errors.empty:
        row = errors.head(N_EXAMPLES_PER_TYPE).copy(); row["example_type"] = "high_conf_error"; row["example_class"] = row["predicted_label"]; selected.append(row)
    if not correct.empty:
        row = correct.head(N_EXAMPLES_PER_TYPE).copy(); row["example_type"] = "correct"; row["example_class"] = row["true_label"]; selected.append(row)
    selected_frame = pd.concat(selected, ignore_index=True) if selected else pd.DataFrame()
    worst = classification_report.copy()
    if not worst.empty and "label" in worst.columns:
        worst = worst[~worst["label"].astype(str).isin(["accuracy", "macro avg", "weighted avg"])]
        sort_cols = [column for column in ["recall", "f1-score"] if column in worst.columns]
        worst = worst.sort_values(sort_cols).head(20) if sort_cols else worst
    return errors, correct, pd.DataFrame(fp_rows), pd.DataFrame(fn_rows), selected_frame, worst


def multi_label_tables(frame):
    true_cols = [column for column in frame.columns if column.startswith("true_")]
    pred_cols = ["pred_" + column.removeprefix("true_") for column in true_cols]
    prob_cols = ["prob_" + column.removeprefix("true_") for column in true_cols]
    data = frame.copy()
    available_prob_cols = [column for column in prob_cols if column in data.columns]
    data["confidence"] = data[available_prob_cols].max(axis=1) if available_prob_cols else 0.0
    data["exact_match_bool"] = data.get("exact_match", False).astype(bool)
    errors = data[~data["exact_match_bool"]].sort_values("confidence", ascending=False)
    correct = data[data["exact_match_bool"]].sort_values("confidence", ascending=False)
    fp_rows = []
    fn_rows = []
    selected = []
    for true_col, pred_col, prob_col in zip(true_cols, pred_cols, prob_cols):
        label = true_col.removeprefix("true_")
        if pred_col not in data.columns:
            continue
        fp = data[(data[true_col] == 0) & (data[pred_col] == 1)].copy()
        fn = data[(data[true_col] == 1) & (data[pred_col] == 0)].copy()
        if prob_col in data.columns:
            fp = fp.sort_values(prob_col, ascending=False)
            fn = fn.assign(false_negative_score=1.0 - fn[prob_col]).sort_values("false_negative_score", ascending=False)
        fp_rows.append({"class": label, "false_positives": len(fp), "mean_probability": float(fp[prob_col].mean()) if prob_col in fp and len(fp) else 0.0})
        fn_rows.append({"class": label, "false_negatives": len(fn), "mean_probability": float(fn[prob_col].mean()) if prob_col in fn and len(fn) else 0.0})
        if not fp.empty:
            row = fp.head(1).copy(); row["example_type"] = "false_positive"; row["example_class"] = label; selected.append(row)
        if not fn.empty:
            row = fn.head(1).copy(); row["example_type"] = "false_negative"; row["example_class"] = label; selected.append(row)
    if not errors.empty:
        row = errors.head(N_EXAMPLES_PER_TYPE).copy(); row["example_type"] = "high_conf_error"; row["example_class"] = "multi_label"; selected.append(row)
    if not correct.empty:
        row = correct.head(N_EXAMPLES_PER_TYPE).copy(); row["example_type"] = "correct"; row["example_class"] = "multi_label"; selected.append(row)
    selected_frame = pd.concat(selected, ignore_index=True) if selected else pd.DataFrame()
    worst = per_class_metrics.copy()
    if not worst.empty:
        sort_cols = [column for column in ["recall", "f1"] if column in worst.columns]
        worst = worst.sort_values(sort_cols).head(20) if sort_cols else worst
    return errors, correct, pd.DataFrame(fp_rows), pd.DataFrame(fn_rows), selected_frame, worst

if TASK_TYPE_RESOLVED == "single_label":
    high_conf_errors, correct_high_conf, false_positive_summary, false_negative_summary, selected_examples, worst_classes = single_label_tables(predictions)
else:
    high_conf_errors, correct_high_conf, false_positive_summary, false_negative_summary, selected_examples, worst_classes = multi_label_tables(predictions)

high_conf_errors.to_csv(ERROR_DIR / "high_confidence_errors.csv", index=False)
correct_high_conf.to_csv(ERROR_DIR / "correct_high_confidence_examples.csv", index=False)
false_positive_summary.to_csv(ERROR_DIR / "false_positive_summary.csv", index=False)
false_negative_summary.to_csv(ERROR_DIR / "false_negative_summary.csv", index=False)
selected_examples.to_csv(ERROR_DIR / "selected_representative_examples.csv", index=False)
worst_classes.to_csv(ERROR_DIR / "worst_classes_by_recall_f1.csv", index=False)

print("Saved error-analysis tables to", ERROR_DIR)
display(selected_examples.head(20))


In [ ]:
def source_audio_exists(row):
    if TASK_TYPE_RESOLVED == "multi_label" and str(row.get("source_audio_paths", "")):
        paths = [value for value in str(row.get("source_audio_paths", "")).split("|") if value]
        return all((MEDLEYDB_ROOT / path).exists() for path in paths)
    audio_path = row.get("audio_path", "")
    return bool(str(audio_path)) and (MEDLEYDB_ROOT / str(audio_path)).exists()


def export_row_audio(row):
    example_type = str(row.get("example_type", "example"))
    example_class = safe_label(row.get("example_class", row.get("predicted_label", "multi_label")))
    row_id = row_identifier(row)
    if example_type == "high_conf_error" and TASK_TYPE_RESOLVED == "single_label":
        filename = f"high_conf_error_true_{safe_label(row.get('true_label'))}_pred_{safe_label(row.get('predicted_label'))}_{row_id}.wav"
    elif example_type in {"false_positive", "false_negative", "correct"}:
        filename = f"{example_type}_{example_class}_{row_id}.wav"
    else:
        filename = f"{example_type}_{example_class}_{row_id}.wav"
    if TASK_TYPE_RESOLVED == "multi_label":
        waveform, sample_rate = load_mixture_segment(row, MEDLEYDB_ROOT, target_sample_rate=TARGET_SAMPLE_RATE)
    else:
        waveform, sample_rate = load_isolated_stem_segment(row, MEDLEYDB_ROOT, target_sample_rate=TARGET_SAMPLE_RATE)
    return export_audio_segment(waveform, sample_rate, AUDIO_DIR / filename)

exported = []
missing_audio = []
if EXPORT_AUDIO and not selected_examples.empty:
    for _, row in selected_examples.iterrows():
        try:
            if not source_audio_exists(row):
                missing_audio.append(row_identifier(row))
                continue
            exported.append(export_row_audio(row))
        except Exception as exc:
            missing_audio.append(f"{row_identifier(row)} ({type(exc).__name__}: {exc})")
    print(f"Exported {len(exported)} audio clips to {AUDIO_DIR}")
else:
    print("Audio export disabled; metadata tables were saved only.")
if missing_audio:
    print("Warning: some selected audio sources were missing or could not be exported:")
    for item in missing_audio[:20]:
        print("-", item)


In [ ]:
print("Automatic interpretation")
print(f"- Experiment: {EXPERIMENT_ID}")
print(f"- Task type: {TASK_TYPE_RESOLVED}")
print(f"- High-confidence/exact-match errors saved: {len(high_conf_errors)}")
print(f"- Correct high-confidence examples saved: {len(correct_high_conf)}")
if not false_positive_summary.empty:
    fp_text = false_positive_summary.sort_values(false_positive_summary.columns[1], ascending=False).head(5).to_dict("records")
    print("- Largest false-positive classes:", fp_text)
if not false_negative_summary.empty:
    fn_text = false_negative_summary.sort_values(false_negative_summary.columns[1], ascending=False).head(5).to_dict("records")
    print("- Largest false-negative classes:", fn_text)
if worst_classes.empty:
    print("- No per-class metric table was found; worst-class CSV is empty.")
else:
    print("- Worst classes table saved from classification_report.csv or per_class_metrics.csv.")
print("- Use selected_representative_examples.csv as the manifest for presentation examples; audio clips are optional and exported only when source files exist.")
